# Time Resampling

Resampling changes the **frequency** of time series data. It is one of the most
common operations you will perform when working with time series in pandas.

There are two directions:

- **Downsampling** — moving from a higher frequency to a lower frequency
  (e.g., daily → monthly). This requires **aggregation** because many data points
  map to one output point.
- **Upsampling** — moving from a lower frequency to a higher frequency
  (e.g., monthly → daily). This introduces **missing values** that must be filled
  or interpolated.

This notebook covers the `resample()` method, modern pandas offset aliases,
downsampling and upsampling strategies, and the difference between `resample()`
and `asfreq()`.

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path("../data")

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

In [ ]:
# Daily stock data — good for downsampling demos
df_sbux = pd.read_csv(
    DATA_DIR / "starbucks.csv",
    parse_dates=True,
    index_col="Date",
)
df_sbux.index.freq = pd.infer_freq(df_sbux.index)  # business-day frequency

print(f"Shape: {df_sbux.shape}")
print(f"Range: {df_sbux.index.min()} → {df_sbux.index.max()}")
print(f"Freq:  {df_sbux.index.freq}")
df_sbux.head()

In [ ]:
# Monthly data — good for showing both down- and upsampling
df_air = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    parse_dates=True,
    index_col="Month",
)
df_air.index.freq = pd.infer_freq(df_air.index)

print(f"Shape: {df_air.shape}")
print(f"Range: {df_air.index.min()} → {df_air.index.max()}")
print(f"Freq:  {df_air.index.freq}")
df_air.head()

---
## 2. The `resample()` Method

The basic syntax is:

```python
df.resample(rule).agg_func()
```

Think of `resample()` as a **time-aware `groupby`**. It groups the data into
time-based buckets defined by the `rule` string, then applies an aggregation
function to each bucket.

| `groupby` analogy | `resample` equivalent |
|---|---|
| `df.groupby('category')` | `df.resample('ME')` |
| `.mean()`, `.sum()`, etc. | `.mean()`, `.sum()`, etc. |

The `rule` argument is an **offset alias** string (see next section).

In [ ]:
# Basic example: daily close prices → monthly mean
monthly_mean = df_sbux['Close'].resample('ME').mean()
monthly_mean.head(10)

---
## 3. Modern Offset Aliases (pandas 2.2+)

Pandas 2.2 **deprecated** many of the old single-letter offset aliases and
introduced clearer replacements. Always use the modern aliases in new code.

### Full Reference Table

| Alias | Meaning |
|---|---|
| `'D'` | Calendar day |
| `'W'` | Week (defaults to Sunday end) |
| `'ME'` | Month **end** |
| `'MS'` | Month **start** |
| `'QE'` | Quarter **end** |
| `'QS'` | Quarter **start** |
| `'YE'` | Year **end** |
| `'YS'` | Year **start** |
| `'h'` | Hour |
| `'min'` | Minute |
| `'s'` | Second |
| `'ms'` | Millisecond |
| `'BME'` | Business month **end** |
| `'BMS'` | Business month **start** |
| `'BQE'` | Business quarter **end** |
| `'BQS'` | Business quarter **start** |
| `'BYE'` | Business year **end** |
| `'BYS'` | Business year **start** |

### Deprecated → Modern Mapping

| Old (deprecated) | New (use this) | Meaning |
|---|---|---|
| `'M'` | `'ME'` | Month end |
| `'Q'` | `'QE'` | Quarter end |
| `'A'` / `'Y'` | `'YE'` | Year end |
| `'H'` | `'h'` | Hour |
| `'T'` | `'min'` | Minute |
| `'S'` | `'s'` | Second |
| `'L'` | `'ms'` | Millisecond |
| `'BM'` | `'BME'` | Business month end |
| `'BQ'` | `'BQE'` | Business quarter end |

> **Rule of thumb:** if the old alias was a single uppercase letter, it has
> probably been replaced. Check the
> [pandas offset aliases docs](https://pandas.pydata.org/docs/user_guide/timeseries.html#dateoffset-objects)
> when in doubt.

---
## 4. Downsampling Examples

Downsampling reduces frequency. Every group of source observations gets
**aggregated** into a single output value.

### 4a. Monthly → Quarterly / Yearly

In [ ]:
# Airline passengers: monthly → quarterly mean
quarterly = df_air.resample('QE').mean()
quarterly.head(8)

In [ ]:
# Monthly → yearly sum (total passengers per year)
yearly = df_air.resample('YE').sum()
yearly

### 4b. Common Aggregation Functions

| Method | What it does |
|---|---|
| `.mean()` | Average of values in each bucket |
| `.sum()` | Total of values in each bucket |
| `.first()` | First observation in each bucket |
| `.last()` | Last observation in each bucket |
| `.min()` | Minimum value in each bucket |
| `.max()` | Maximum value in each bucket |
| `.ohlc()` | Open, High, Low, Close (financial data) |
| `.agg()` | Apply one or more custom aggregations |

In [ ]:
# Multiple aggregations at once
df_sbux['Close'].resample('QE').agg(['mean', 'std', 'min', 'max']).head(8)

In [ ]:
# Named aggregation (pandas >= 0.25)
df_sbux.resample('YE').agg(
    avg_close=('Close', 'mean'),
    total_volume=('Volume', 'sum'),
    max_close=('Close', 'max'),
)

### 4c. OHLC Resampling

The `.ohlc()` method returns **Open, High, Low, Close** for each period —
exactly what you need for candlestick-style financial analysis.

In [ ]:
ohlc = df_sbux['Close'].resample('ME').ohlc()
ohlc.head(10)

### 4d. Custom Function with `.apply()`

In [ ]:
# Custom aggregation: range (max - min) within each quarter
def price_range(series):
    return series.max() - series.min()

df_sbux['Close'].resample('QE').apply(price_range).head(8)

### 4e. Visualize Original vs Resampled Data

In [ ]:
monthly_close = df_sbux['Close'].resample('ME').mean()
quarterly_close = df_sbux['Close'].resample('QE').mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_sbux['Close'], alpha=0.4, linewidth=0.8, label='Daily')
ax.plot(monthly_close, linewidth=1.5, label='Monthly mean')
ax.plot(quarterly_close, linewidth=2.2, label='Quarterly mean', linestyle='--')
ax.set_title('Starbucks Close Price — Downsampled Views')
ax.set_ylabel('Price ($)')
ax.legend()
plt.tight_layout()
plt.show()

Notice how downsampling smooths out the day-to-day noise, making the overall
trend easier to see. The quarterly line is the smoothest because it averages
over the largest windows.

---
## 5. Upsampling

Upsampling moves from a **lower** to a **higher** frequency. This creates new
time points that don't have corresponding observations — so we need a strategy
to fill in the gaps.

In [ ]:
# Start with a small monthly slice for clarity
monthly_slice = df_air['Thousands of Passengers']['1949-01':'1949-06']
print("Original (monthly):")
print(monthly_slice)
print()

In [ ]:
# Upsample to daily — creates NaN at new date points
daily_raw = monthly_slice.resample('D').asfreq()
print(f"Monthly points: {len(monthly_slice)}")
print(f"Daily points:   {len(daily_raw)}")
print(f"NaN count:      {daily_raw.isna().sum()}")
daily_raw.head(10)

### Fill Methods for Upsampled Data

| Method | Behaviour |
|---|---|
| `.asfreq()` | No fill — leaves NaN |
| `.ffill()` | Forward fill — carry the last known value forward |
| `.bfill()` | Backward fill — use the next known value |
| `.interpolate()` | Linear (or other) interpolation between known points |

In [ ]:
daily_ffill  = monthly_slice.resample('D').ffill()
daily_bfill  = monthly_slice.resample('D').bfill()
daily_interp = monthly_slice.resample('D').interpolate()

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, data, title in zip(
    axes,
    [daily_ffill, daily_bfill, daily_interp],
    ['Forward Fill', 'Backward Fill', 'Interpolate'],
):
    ax.plot(data, linewidth=1, color='steelblue')
    ax.plot(monthly_slice, 'o', color='tomato', markersize=6, label='Original')
    ax.set_title(title)
    ax.legend()

fig.suptitle('Upsampling Monthly → Daily: Fill Strategies', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**When does upsampling make sense?**

- When you need to **align** two series that have different frequencies (e.g.,
  merge monthly economic data with daily stock prices).
- When a downstream model or visualization requires a specific frequency.

**When it does not make sense:**

- Upsampling does not create new information. Forward fill just repeats values;
  interpolation assumes smooth transitions that may not reflect reality.
- Avoid upsampling by large factors (e.g., yearly → hourly) as this produces
  mostly synthetic data.

---
## 6. `asfreq()` vs `resample()`

These two methods both change frequency, but they work differently:

| | `asfreq()` | `resample()` |
|---|---|---|
| **What it does** | Selects a single value at each new frequency point | Groups all values within each period, then aggregates |
| **Aggregation** | None — just picks or creates NaN | Required (mean, sum, etc.) |
| **Use case** | Thinning data, converting frequency | Computing statistics over time windows |

In [ ]:
# Side-by-side comparison: daily → monthly
close = df_sbux['Close']

asfreq_monthly   = close.asfreq('ME')          # picks the value on the last day of month
resample_monthly = close.resample('ME').mean()  # averages all daily values in that month

comparison = pd.DataFrame({
    'asfreq (single value)': asfreq_monthly,
    'resample mean': resample_monthly,
})
comparison.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(close, alpha=0.3, linewidth=0.7, label='Daily (original)', color='gray')
ax.plot(asfreq_monthly, 'o-', markersize=3, linewidth=1.2, label='asfreq (month-end value)')
ax.plot(resample_monthly, 's-', markersize=3, linewidth=1.2, label='resample mean')
ax.set_title('asfreq() vs resample() — Starbucks Close Price')
ax.set_ylabel('Price ($)')
ax.legend()
plt.tight_layout()
plt.show()

The `asfreq` line is more jagged because it takes a single snapshot at each
month end, while the `resample` mean smooths over the entire month.
Choose `asfreq()` when you want a specific point-in-time reading;
choose `resample()` when you want a summary statistic.

---
## 7. Practical Example: Energy Production Index

Let's tie everything together with the Energy Production dataset. We will
resample from monthly to quarterly and yearly frequencies, then compare all
three visually.

In [ ]:
df_energy = pd.read_csv(
    DATA_DIR / "EnergyProduction.csv",
    parse_dates=True,
    index_col="DATE",
)
df_energy.index.freq = pd.infer_freq(df_energy.index)

print(f"Shape: {df_energy.shape}")
print(f"Range: {df_energy.index.min()} → {df_energy.index.max()}")
print(f"Freq:  {df_energy.index.freq}")
df_energy.head()

In [ ]:
# Resample to quarterly and yearly
energy_quarterly = df_energy.resample('QE').mean()
energy_yearly    = df_energy.resample('YE').mean()

print(f"Monthly rows:    {len(df_energy)}")
print(f"Quarterly rows:  {len(energy_quarterly)}")
print(f"Yearly rows:     {len(energy_yearly)}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df_energy, alpha=0.4, linewidth=0.8, label='Monthly', color='steelblue')
ax.plot(energy_quarterly, linewidth=1.5, label='Quarterly mean', color='darkorange')
ax.plot(energy_yearly, linewidth=2.5, label='Yearly mean', color='firebrick',
        linestyle='--')

ax.set_title('Energy Production Index — Three Frequencies')
ax.set_ylabel('Energy Index')
ax.legend()
plt.tight_layout()
plt.show()

### Year-over-Year Totals

In [ ]:
# Year-over-year totals using resample
yearly_totals = df_energy.resample('YE').sum()
yearly_totals.columns = ['YearlyTotal']

# Calculate year-over-year percent change
yearly_totals['YoY_Change_%'] = yearly_totals['YearlyTotal'].pct_change() * 100
yearly_totals.dropna().head(10)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: yearly totals as bar chart
axes[0].bar(yearly_totals.index.year, yearly_totals['YearlyTotal'],
            color='steelblue', alpha=0.8)
axes[0].set_ylabel('Yearly Total')
axes[0].set_title('Energy Production — Annual Totals & Year-over-Year Change')

# Bottom: YoY % change
yoy = yearly_totals['YoY_Change_%'].dropna()
colors = ['firebrick' if v < 0 else 'seagreen' for v in yoy]
axes[1].bar(yoy.index.year, yoy, color=colors, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('YoY Change (%)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.show()

---
## Summary

| Concept | Key Takeaway |
|---|---|
| `resample()` | Time-aware `groupby` — groups data into time buckets, then aggregates |
| Downsampling | Higher → lower frequency; requires aggregation (mean, sum, etc.) |
| Upsampling | Lower → higher frequency; introduces NaN, fill with `ffill`, `bfill`, or `interpolate` |
| Modern aliases | Use `'ME'`, `'QE'`, `'YE'`, `'h'`, `'min'`, `'s'` — not the deprecated `'M'`, `'Q'`, `'A'`, `'H'`, `'T'`, `'S'` |
| `asfreq()` vs `resample()` | `asfreq` picks a single value; `resample` aggregates all values in the period |
| OHLC | `.ohlc()` returns Open/High/Low/Close — useful for financial data |
| `.agg()` | Apply multiple or named aggregations in one call |